In [ ]:
NCAA_tourney_results_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv'
reg_season_results_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/WRegularSeasonDetailedResults.csv'
tourney_seeds_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/WNCAATourneySeeds.csv'
teams_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/WTeams.csv'
conference_games_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/WConferenceTourneyGames.csv'
seasons_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/WSeasons.csv'
team_conferences_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/WTeamConferences.csv'

In [ ]:
import pandas as pd

NCAA_results = pd.read_csv(NCAA_tourney_results_path)
reg_season_results = pd.read_csv(reg_season_results_path)
tourney_seeds = pd.read_csv(tourney_seeds_path)
teams = pd.read_csv(teams_path)
conference_games = pd.read_csv(conference_games_path)
seasons = pd.read_csv(seasons_path)
team_conferences = pd.read_csv(team_conferences_path)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

base_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/'

NCAA_results       = pd.read_csv(base_path + 'WNCAATourneyDetailedResults.csv')
reg_season_results = pd.read_csv(base_path + 'WRegularSeasonDetailedResults.csv')
tourney_seeds      = pd.read_csv(base_path + 'WNCAATourneySeeds.csv')
teams              = pd.read_csv(base_path + 'WTeams.csv')
conference_games   = pd.read_csv(base_path + 'WConferenceTourneyGames.csv')

# BUILD TEAM STATS 
def build_team_stats(df):
    winners = df.rename(columns={
        'WTeamID': 'TeamID', 'WScore': 'Score', 'LScore': 'OppScore',
        'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3': 'FGM3', 'WFGA3': 'FGA3',
        'WFTM': 'FTM', 'WFTA': 'FTA', 'WOR': 'OR', 'WDR': 'DR',
        'WAst': 'Ast', 'WTO': 'TO', 'WStl': 'Stl', 'WBlk': 'Blk'
    })
    winners['Won'] = 1

    losers = df.rename(columns={
        'LTeamID': 'TeamID', 'LScore': 'Score', 'WScore': 'OppScore',
        'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3', 'LFGA3': 'FGA3',
        'LFTM': 'FTM', 'LFTA': 'FTA', 'LOR': 'OR', 'LDR': 'DR',
        'LAst': 'Ast', 'LTO': 'TO', 'LStl': 'Stl', 'LBlk': 'Blk'
    })
    losers['Won'] = 0

    return pd.concat([winners, losers], ignore_index=True)

reg_all = build_team_stats(reg_season_results)

team_stats = reg_all.groupby(['Season', 'TeamID']).agg(
    Win_Pct       = ('Won',      'mean'),
    Avg_Score     = ('Score',    'mean'),
    Avg_OppScore  = ('OppScore', 'mean'),
    Avg_FGM3      = ('FGM3',     'mean'),
    Avg_FGA3      = ('FGA3',     'mean'),
    Avg_FGM       = ('FGM',      'mean'),
    Avg_FGA       = ('FGA',      'mean'),
    Avg_FTM       = ('FTM',      'mean'),
    Avg_FTA       = ('FTA',      'mean'),
    Avg_Ast       = ('Ast',      'mean'),
    Avg_TO        = ('TO',       'mean'),
    Avg_Stl       = ('Stl',      'mean'),
    Avg_Blk       = ('Blk',      'mean'),
    Avg_OR        = ('OR',       'mean'),
    Avg_DR        = ('DR',       'mean'),
).reset_index()

# features
team_stats['Scoring_Margin'] = team_stats['Avg_Score']  - team_stats['Avg_OppScore']
team_stats['Ast_TO_Ratio']   = team_stats['Avg_Ast']    / team_stats['Avg_TO']
team_stats['Three_PT_Pct']   = team_stats['Avg_FGM3']   / team_stats['Avg_FGA3']
team_stats['Two_PT_Pct']     = (team_stats['Avg_FGM']  - team_stats['Avg_FGM3']) / \
                                (team_stats['Avg_FGA']  - team_stats['Avg_FGA3'])
team_stats['True_Shooting']  = team_stats['Avg_Score']  / \
                                (2 * (team_stats['Avg_FGA'] + 0.44 * team_stats['Avg_FTA']))
team_stats['TO_Margin']      = team_stats['Avg_Stl']    - team_stats['Avg_TO']
team_stats['Total_Rebounds'] = team_stats['Avg_OR']     + team_stats['Avg_DR']

tourney_seeds['Seed_Num'] = tourney_seeds['Seed'].str[1:]\
    .str.replace('a','').str.replace('b','').astype(int)

team_stats = team_stats.merge(
    tourney_seeds[['Season', 'TeamID', 'Seed_Num']],
    on=['Season', 'TeamID'], how='left'
)

print(f"Team stats shape (all seasons): {team_stats.shape}")
print(f"Seasons covered: {sorted(team_stats['Season'].unique())[:5]} ... "
      f"{sorted(team_stats['Season'].unique())[-5:]}")

# BUILD MATCHUP DATASETS
features = [
    'Seed_Num', 'Win_Pct', 'Scoring_Margin', 'Avg_Score',
    'Avg_OppScore', 'Three_PT_Pct', 'Two_PT_Pct', 'True_Shooting',
    'Ast_TO_Ratio', 'TO_Margin', 'Total_Rebounds', 'Avg_Blk', 'Avg_Stl'
]

def build_matchup_from_compact(results_df, stats_df, season_filter=None):
    if season_filter == 'train':
        results_df = results_df[results_df['Season'] < 2024]
    elif season_filter == 'test':
        results_df = results_df[results_df['Season'] >= 2024]

    rows = []
    for _, game in results_df.iterrows():
        season = game['Season']
        w_team = game['WTeamID']
        l_team = game['LTeamID']

        w_stats = stats_df[(stats_df['Season'] == season) &
                           (stats_df['TeamID'] == w_team)]
        l_stats = stats_df[(stats_df['Season'] == season) &
                           (stats_df['TeamID'] == l_team)]

        if w_stats.empty or l_stats.empty:
            continue

        w = w_stats.iloc[0]
        l = l_stats.iloc[0]

        row = {'Season': season}
        for f in features:
            row[f'T1_{f}'] = w[f] if f in w.index else np.nan
            row[f'T2_{f}'] = l[f] if f in l.index else np.nan
        row['T1_TeamID'] = w_team
        row['T2_TeamID'] = l_team
        row['Result'] = 1

        row_flip = {'Season': season}
        for f in features:
            row_flip[f'T1_{f}'] = l[f] if f in l.index else np.nan
            row_flip[f'T2_{f}'] = w[f] if f in w.index else np.nan
        row_flip['T1_TeamID'] = l_team
        row_flip['T2_TeamID'] = w_team
        row_flip['Result'] = 0

        rows.append(row)
        rows.append(row_flip)

    return pd.DataFrame(rows)

# build all three training sources 
print("\nBuilding training matchups from all seasons before 2024...")
ncaa_train = build_matchup_from_compact(NCAA_results, team_stats, 'train')
conf_train = build_matchup_from_compact(conference_games, team_stats, 'train')
reg_train  = build_matchup_from_compact(reg_season_results, team_stats, 'train')

print(f"NCAA tournament matchups:     {len(ncaa_train):>8,} rows")
print(f"Conference tournament matchups:{len(conf_train):>7,} rows")
print(f"Regular season matchups:      {len(reg_train):>8,} rows")

train_matchups = pd.concat([ncaa_train, conf_train, reg_train], ignore_index=True)
print(f"\nTotal training matchups:      {len(train_matchups):>8,} rows")

# test
test_matchups = build_matchup_from_compact(NCAA_results, team_stats, 'test')
print(f"Test matchups:                {len(test_matchups):>8,} rows")

# TRAIN MODELS
matchup_features = [f'T1_{f}' for f in features] + [f'T2_{f}' for f in features]

X_train = train_matchups[matchup_features].fillna(0)
y_train = train_matchups['Result']
X_test  = test_matchups[matchup_features].fillna(0)
y_test  = test_matchups['Result']

print(f"\nFinal train size: {len(X_train):,} | Test size: {len(X_test)}")

# scale for logistic regression
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# logistic regression
print("\nTraining Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_preds = lr.predict(X_test_sc)
lr_acc   = accuracy_score(y_test, lr_preds)
lr_f1    = f1_score(y_test, lr_preds, average='weighted')

# random forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_preds)
rf_f1    = f1_score(y_test, rf_preds, average='weighted')

# xgboost
print("Training XGBoost...")
xgb = XGBClassifier(n_estimators=200, random_state=42,
                     eval_metric='logloss', verbosity=0)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_acc   = accuracy_score(y_test, xgb_preds)
xgb_f1    = f1_score(y_test, xgb_preds, average='weighted')

print("\n" + "="*55)
print("MODEL RESULTS — TRAINED ON ALL SEASONS < 2024")
print("="*55)
print(f"{'Model':<25} {'Accuracy':>10} {'F1 Score':>10}")
print("-"*55)
print(f"{'Logistic Regression':<25} {lr_acc:>10.4f} {lr_f1:>10.4f}")
print(f"{'Random Forest':<25} {rf_acc:>10.4f} {rf_f1:>10.4f}")
print(f"{'XGBoost':<25} {xgb_acc:>10.4f} {xgb_f1:>10.4f}")

# SIMULATE TOURNAMENTS
def predict_game(team1_id, team2_id, season, model, stats_df, scaler=None):
    t1 = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team1_id)]
    t2 = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team2_id)]

    if t1.empty or t2.empty:
        return team1_id, 0.5

    row = {}
    for f in features:
        row[f'T1_{f}'] = t1.iloc[0][f] if f in t1.columns else 0
        row[f'T2_{f}'] = t2.iloc[0][f] if f in t2.columns else 0

    X = pd.DataFrame([row])[matchup_features].fillna(0)
    if scaler:
        X = scaler.transform(X)

    pred = model.predict(X)[0]
    prob = model.predict_proba(X)[0]
    winner   = team1_id if pred == 1 else team2_id
    win_prob = prob[1]   if pred == 1 else prob[0]
    return winner, win_prob


def simulate_tournament(season, model, stats_df, scaler=None):
    seeds = tourney_seeds[tourney_seeds['Season'] == season].copy()
    seeds['Seed_Num'] = seeds['Seed'].str[1:]\
        .str.replace('a','').str.replace('b','').astype(int)
    seeds = seeds.merge(teams[['TeamID', 'TeamName']], on='TeamID', how='left')
    seeds = seeds.sort_values('Seed_Num')

    team_list = seeds[['TeamID', 'TeamName', 'Seed_Num']]\
        .drop_duplicates('TeamID').values.tolist()

    round_names = {1: 'Round of 64', 2: 'Round of 32',
                   3: 'Sweet 16',    4: 'Elite 8',
                   5: 'Final Four',  6: 'Championship'}

    print(f"\n{'='*60}")
    print(f"TOURNAMENT SIMULATION — {season} SEASON")
    print(f"{'='*60}")

    current_teams = team_list
    round_num     = 1

    while len(current_teams) > 1:
        round_name = round_names.get(round_num, f'Round {round_num}')
        print(f"\n--- {round_name} ---")
        next_round = []

        mid      = len(current_teams) // 2
        matchups = list(zip(current_teams[:mid], current_teams[mid:][::-1]))

        for t1, t2 in matchups:
            winner_id, prob = predict_game(
                t1[0], t2[0], season, model, stats_df, scaler
            )
            winner_info = t1 if winner_id == t1[0] else t2
            print(f"  #{t1[2]} {t1[1]:<28} vs "
                  f"#{t2[2]} {t2[1]:<28} → "
                  f"{winner_info[1]} ({prob:.0%})")
            next_round.append(winner_info)

        current_teams = next_round
        round_num    += 1

    champion = current_teams[0]
    print(f"\n{'='*60}")
    print(f"PREDICTED CHAMPION: #{champion[2]} {champion[1]}")
    print(f"{'='*60}")
    return champion

# simulate both seasons
print("\nSimulating 2025 with Logistic Regression:")
champ_2025_lr = simulate_tournament(2025, lr, team_stats, scaler)

print("\nSimulating 2026 with Logistic Regression:")
champ_2026_lr = simulate_tournament(2026, lr, team_stats, scaler)

print("\nSimulating 2025 with Random Forest:")
champ_2025_rf = simulate_tournament(2025, rf, team_stats)

print("\nSimulating 2026 with Random Forest:")
champ_2026_rf = simulate_tournament(2026, rf, team_stats)

# PERFECT BRACKET MATH
best_acc = max(lr_acc, rf_acc, xgb_acc)
print(f"\n{'='*50}")
print("PERFECT BRACKET ANALYSIS")
print(f"{'='*50}")
print(f"Best model accuracy:    {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"P(perfect bracket):     {best_acc**63:.2e}")
print(f"≈ 1 in {int(1/best_acc**63):,}")
print(f"Random chance:          {0.5**63:.2e}")
print(f"≈ 1 in {int(1/0.5**63):,}")

In [ ]:
# MONTE CARLO SIMULATION 
def simulate_tournament_probabilistic(season, model, stats_df, scaler=None):
    seeds = tourney_seeds[tourney_seeds['Season'] == season].copy()
    seeds['Seed_Num'] = seeds['Seed'].str[1:]\
        .str.replace('a','').str.replace('b','').astype(int)
    seeds = seeds.merge(teams[['TeamID', 'TeamName']], on='TeamID', how='left')
    seeds = seeds.sort_values('Seed_Num')

    team_list = seeds[['TeamID', 'TeamName', 'Seed_Num']]\
        .drop_duplicates('TeamID').values.tolist()

    current_teams = team_list
    round_num = 1

    while len(current_teams) > 1:
        next_round = []
        mid      = len(current_teams) // 2
        matchups = list(zip(current_teams[:mid], current_teams[mid:][::-1]))

        for t1, t2 in matchups:
            winner_id, prob = predict_game(
                t1[0], t2[0], season, model, stats_df, scaler
            )
            # use probability to randomly pick winner
            if np.random.random() < prob:
                winner_info = t1 if winner_id == t1[0] else t2
            else:
                winner_info = t2 if winner_id == t1[0] else t1

            next_round.append(winner_info)

        current_teams = next_round
        round_num += 1

    return current_teams[0]


def monte_carlo_tournament(season, model, stats_df,
                            n_simulations=1000, scaler=None):
    champion_counts = {}

    for i in range(n_simulations):
        np.random.seed(i)
        champ      = simulate_tournament_probabilistic(
            season, model, stats_df, scaler
        )
        champ_name = champ[1]
        champion_counts[champ_name] = champion_counts.get(champ_name, 0) + 1

    results = pd.DataFrame([
        {
            'Team': team,
            'Championships': count,
            'Win_Probability': count / n_simulations * 100
        }
        for team, count in champion_counts.items()
    ]).sort_values('Win_Probability', ascending=False).reset_index(drop=True)

    return results

# RUN 1000 SIMULATIONS — 2025 AND 2026
print("Running 1000 Monte Carlo simulations for 2025...")
mc_2025_lr = monte_carlo_tournament(
    2025, lr, team_stats, n_simulations=1000, scaler=scaler
)

print("Running 1000 Monte Carlo simulations for 2026...")
mc_2026_lr = monte_carlo_tournament(
    2026, lr, team_stats, n_simulations=1000, scaler=scaler
)

# RESULTS
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2025
top_2025 = mc_2025_lr.head(10)
colors_2025 = ['#2ecc71' if t == 'Connecticut' else 'steelblue'
               for t in top_2025['Team']]
axes[0].barh(top_2025['Team'][::-1], top_2025['Win_Probability'][::-1],
             color=colors_2025[::-1])
axes[0].axvline(
    x=mc_2025_lr[mc_2025_lr['Team'] == 'Connecticut']['Win_Probability'].values[0]
    if 'Connecticut' in mc_2025_lr['Team'].values else 0,
    color='green', linestyle='--', linewidth=2,
    label='Actual Champion (UConn)'
)
axes[0].set_xlabel('Championship Win Probability (%)')
axes[0].set_title('2025 Tournament — 1,000 Simulations\n(Logistic Regression)')
axes[0].legend(fontsize=9)

# 2026
top_2026 = mc_2026_lr.head(10)
colors_2026 = ['#2ecc71' if t == 'UCLA' else 'coral'
               for t in top_2026['Team']]
axes[1].barh(top_2026['Team'][::-1], top_2026['Win_Probability'][::-1],
             color=colors_2026[::-1])
axes[1].axvline(
    x=mc_2026_lr[mc_2026_lr['Team'] == 'UCLA']['Win_Probability'].values[0]
    if 'UCLA' in mc_2026_lr['Team'].values else 0,
    color='green', linestyle='--', linewidth=2,
    label='Actual Champion (UCLA)'
)
axes[1].set_xlabel('Championship Win Probability (%)')
axes[1].set_title('2026 Tournament — 1,000 Simulations\n(Logistic Regression)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig4_monte_carlo.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig4_monte_carlo.png")


print("\n2025 Tournament — Top 10 Predicted Champions:")
print(mc_2025_lr.head(10).to_string(index=False))

uconn_prob = mc_2025_lr[mc_2025_lr['Team'] == 'Connecticut']['Win_Probability']
if not uconn_prob.empty:
    print(f"\nActual 2025 Champion (UConn) predicted to win: "
          f"{uconn_prob.values[0]:.1f}% of simulations")

print("\n2026 Tournament — Top 10 Predicted Champions:")
print(mc_2026_lr.head(10).to_string(index=False))

ucla_prob = mc_2026_lr[mc_2026_lr['Team'] == 'UCLA']['Win_Probability']
if not ucla_prob.empty:
    print(f"\nActual 2026 Champion (UCLA) predicted to win: "
          f"{ucla_prob.values[0]:.1f}% of simulations")

In [ ]:
import pandas as pd
import numpy as np

base_path = '/Users/isabeladelacruz/Downloads/march-machine-learning-mania-2026/'

# reload files
seasons        = pd.read_csv(base_path + 'WSeasons.csv')
tourney_seeds2 = pd.read_csv(base_path + 'WNCAATourneySeeds.csv')
teams2         = pd.read_csv(base_path + 'WTeams.csv')
team_conferences = pd.read_csv(base_path + 'WTeamConferences.csv')
reg_season_results2 = pd.read_csv(base_path + 'WRegularSeasonDetailedResults.csv')
NCAA_results2  = pd.read_csv(base_path + 'WNCAATourneyDetailedResults.csv')

# filter to 2016+
seasons = seasons[seasons['Season'] >= 2016]

# reshape reg season
def build_team_stats_eda(df):
    winners = df.rename(columns={
        'WTeamID': 'TeamID', 'WScore': 'Score', 'LScore': 'OppScore',
        'WFGM3': 'FGM3', 'WFGA3': 'FGA3', 'WAst': 'Ast',
        'WTO': 'TO', 'WStl': 'Stl', 'WBlk': 'Blk',
        'WOR': 'OR', 'WDR': 'DR'
    })
    winners['Won'] = 1
    losers = df.rename(columns={
        'LTeamID': 'TeamID', 'LScore': 'Score', 'WScore': 'OppScore',
        'LFGM3': 'FGM3', 'LFGA3': 'FGA3', 'LAst': 'Ast',
        'LTO': 'TO', 'LStl': 'Stl', 'LBlk': 'Blk',
        'LOR': 'OR', 'LDR': 'DR'
    })
    losers['Won'] = 0
    return pd.concat([winners, losers], ignore_index=True)

reg_all2 = build_team_stats_eda(reg_season_results2)
reg_all2 = reg_all2[reg_all2['Season'] >= 2016]

reg_season_avgs2 = reg_all2.groupby(['Season', 'TeamID']).agg(
    Win_Pct     = ('Won',      'mean'),
    Avg_Score   = ('Score',    'mean'),
    Avg_OppScore= ('OppScore', 'mean'),
    Avg_FGM3    = ('FGM3',     'mean'),
    Avg_FGA3    = ('FGA3',     'mean'),
    Avg_Ast     = ('Ast',      'mean'),
    Avg_TO      = ('TO',       'mean'),
    Avg_Stl     = ('Stl',      'mean'),
    Avg_Blk     = ('Blk',      'mean'),
    Avg_OR      = ('OR',       'mean'),
    Avg_DR      = ('DR',       'mean'),
    Games_Played= ('Won',      'count')
).reset_index()

reg_season_avgs2['Avg_3PT_Pct'] = reg_season_avgs2['Avg_FGM3'] / reg_season_avgs2['Avg_FGA3']
reg_season_avgs2['Scoring_Margin'] = reg_season_avgs2['Avg_Score'] - reg_season_avgs2['Avg_OppScore']

# add seeds
tourney_seeds2['Seed_Num'] = tourney_seeds2['Seed'].str[1:]\
    .str.replace('a','').str.replace('b','').astype(int)

# build tournament wins target
ncaa_w = NCAA_results2.rename(columns={'WTeamID': 'TeamID'})
ncaa_w['Won'] = 1
ncaa_l = NCAA_results2.rename(columns={'LTeamID': 'TeamID'})
ncaa_l['Won'] = 0
ncaa_all2 = pd.concat([ncaa_w[['Season','TeamID','Won']],
                        ncaa_l[['Season','TeamID','Won']]], ignore_index=True)
ncaa_all2 = ncaa_all2[ncaa_all2['Season'] >= 2016]
tourney_wins2 = ncaa_all2.groupby(['Season','TeamID']).agg(
    Tourney_Wins=('Won','sum')).reset_index()

# merge everything
seasons_tourney2 = pd.merge(seasons, tourney_seeds2, on='Season', how='inner')
teams_seasons2   = pd.merge(seasons_tourney2, teams2, on='TeamID', how='left')
merge_conf2      = pd.merge(teams_seasons2, team_conferences,
                            on=['Season','TeamID'], how='left')
merge_reg2       = pd.merge(merge_conf2, reg_season_avgs2,
                            on=['Season','TeamID'], how='left')
MasterDataset    = pd.merge(merge_reg2, tourney_wins2,
                            on=['Season','TeamID'], how='left')
MasterDataset['Tourney_Wins'] = MasterDataset['Tourney_Wins'].fillna(0)
MasterDataset = MasterDataset.drop(
    columns=['DayZero','RegionW','RegionX','RegionY','RegionZ'], errors='ignore'
)

# add Seed_Num and Round_Label
MasterDataset['Seed_Num'] = MasterDataset['Seed'].str[1:]\
    .str.replace('a','').str.replace('b','').astype(int)

def label_round(wins):
    if wins == 0:   return 'R1 Exit'
    elif wins == 1: return 'R2 Exit'
    elif wins == 2: return 'Sweet 16'
    elif wins == 3: return 'Elite 8'
    elif wins == 4: return 'Final Four'
    elif wins == 5: return 'Runner Up'
    else:           return 'Champion'

MasterDataset['Round_Label'] = MasterDataset['Tourney_Wins'].apply(label_round)

print("MasterDataset shape:", MasterDataset.shape)
print("Ready to generate figures!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 11

# FIGURE 1 — Feature Correlation Bar Chart
corr_data = {
    'Seed_Num': 0.656963,
    'Avg_Score': 0.372694,
    'Win_Pct': 0.352233,
    'Avg_Blk': 0.344009,
    'Avg_Ast': 0.321564,
    'Avg_DR': 0.282553,
    'Avg_3PT_Pct': 0.239943,
    'Avg_OR': 0.171001,
    'Avg_TO': 0.130297,
    'Avg_FGA3': 0.136053,
    'Avg_OppScore': 0.101165,
    'Games_Played': 0.087745,
    'Avg_Stl': 0.025372,
}
corr_series = pd.Series(corr_data).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(corr_series.index, corr_series.values, color='steelblue')
plt.axhline(y=0.3, color='red', linestyle='--', linewidth=1.5,
            label='0.3 threshold (moderate)')
plt.axhline(y=0.5, color='darkred', linestyle='--', linewidth=1.5,
            label='0.5 threshold (strong)')
plt.title('Feature Correlation with Tournament Wins (Absolute Value)')
plt.ylabel('Absolute Correlation Coefficient')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('fig1_feature_correlation.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig1_feature_correlation.png")

# FIGURE 2 — 3PT% by Round Reached
plt.figure(figsize=(10, 5))
order = ['R1 Exit', 'R2 Exit', 'Sweet 16', 'Elite 8',
         'Final Four', 'Runner Up', 'Champion']
sns.boxplot(data=MasterDataset, x='Round_Label', y='Avg_3PT_Pct',
            order=order, palette='Blues')
plt.title('3PT% by Tournament Round Reached')
plt.xlabel('Round Reached')
plt.ylabel('Average 3PT%')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('fig2_3pt_by_round.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig2_3pt_by_round.png")

# FIGURE 3 — Model Performance Comparison
models     = ['Logistic Regression', 'Random Forest', 'XGBoost']
accuracies = [0.6530, 0.7985, 0.8022]
f1_scores  = [0.6530, 0.7985, 0.8022]

x     = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, accuracies, width,
               label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, f1_scores,  width,
               label='F1 Score',  color='coral')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title("Model Performance Comparison\nNCCA Women's Tournament Prediction")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1)
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=10,
            fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=10,
            fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_model_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig3_model_comparison.png")

# FIGURE 4 — Monte Carlo Simulation
mc_2025 = {
    'Connecticut': 16.6, 'UCLA': 13.2, 'TCU': 13.1,
    'South Carolina': 8.4, 'USC': 6.9, 'Texas': 5.7,
    'Notre Dame': 4.4, 'Grand Canyon': 3.7,
    'Fairfield': 2.8, 'George Mason': 2.2
}
mc_2026 = {
    'Connecticut': 34.9, 'UCLA': 19.6, 'South Carolina': 15.0,
    'LSU': 8.6, 'Texas': 8.3, 'Fairfield': 1.8,
    'Vanderbilt': 1.5, 'Michigan': 1.3,
    'Murray St': 0.9, 'Iowa': 0.8
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2025
teams_25 = list(mc_2025.keys())
probs_25 = list(mc_2025.values())
colors_25 = ['#2ecc71' if t == 'Connecticut' else 'steelblue'
             for t in teams_25]
axes[0].barh(teams_25[::-1], probs_25[::-1], color=colors_25[::-1])
axes[0].axvline(x=16.6, color='green', linestyle='--',
                linewidth=2, label='Actual Champion (UConn)')
axes[0].set_xlabel('Championship Win Probability (%)')
axes[0].set_title('2025 Tournament — 1,000 Simulations\n(Logistic Regression)')
axes[0].legend(fontsize=9)
for i, p in enumerate(probs_25[::-1]):
    axes[0].text(p + 0.2, i, f'{p:.1f}%', va='center', fontsize=9)

# 2026
teams_26 = list(mc_2026.keys())
probs_26 = list(mc_2026.values())
colors_26 = ['#2ecc71' if t == 'UCLA' else 'coral' for t in teams_26]
axes[1].barh(teams_26[::-1], probs_26[::-1], color=colors_26[::-1])
axes[1].axvline(x=19.6, color='green', linestyle='--',
                linewidth=2, label='Actual Champion (UCLA)')
axes[1].set_xlabel('Championship Win Probability (%)')
axes[1].set_title('2026 Tournament — 1,000 Simulations\n(Logistic Regression)')
axes[1].legend(fontsize=9)
for i, p in enumerate(probs_26[::-1]):
    axes[1].text(p + 0.2, i, f'{p:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig4_monte_carlo.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig4_monte_carlo.png")

# FIGURE 5 — Feature Importances (Random Forest)
rf_importances = {
    'Seed_Num':        0.234370,
    'Scoring_Margin':  0.120193,
    'Off_Rating':      0.103214,
    'Ast_TO_Ratio':    0.095820,
    'Win_Pct':         0.094062,
    'Total_Rebounds':  0.091415,
    'Three_PT_Pct':    0.089451,
    'Blocks_Per_Game': 0.087975,
    'TO_Margin':       0.083499,
}
rf_series = pd.Series(rf_importances).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
bars = plt.bar(rf_series.index, rf_series.values, color='steelblue')
plt.axhline(y=0.10, color='red', linestyle='--', linewidth=1.5,
            label='10% importance threshold')
plt.title("Feature Importances — Random Forest\nNCCA Women's Tournament Prediction")
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.legend()
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.002,
             f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig5_feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved fig5_feature_importance.png")

print("\n All 5 figures saved! Now upload to Overleaf.")